[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditya-siraskar/PINN_FTRE/blob/main/notebooks/05_phase2_stage4_pinn_nd.ipynb)

# 05 · Stage 4: multi-asset PINN and scaling study

**Phase:** Phase 2: multi-asset

**Goal:** PINN for the d-dimensional common-jump PIDE at d = 1, 2, 5, 10: joint-density and portfolio VaR/ES accuracy against the exact mixture, Monte Carlo at the same budget, and training time and memory vs d (RQ1, RQ4).

**Outputs:** `figures/stage4_pinn_nd_d<d>/fig01–fig04`, `figures/stage4_summary/`, `results/stage4_*`

> Runtime → Change runtime type → **T4 GPU**. Run the setup cell first. With `USE_DRIVE = True` the repo lives in
> `MyDrive/PINN_FTRE`, so figures and results persist. Core logic lives in `pinn_tailrisk/` and `scripts/`;
> this notebook only runs it and shows the results.

In [ ]:
# @title Setup: repo on Google Drive, install, GPU check  { display-mode: "form" }
USE_DRIVE = True   # keep figures/results across Colab sessions (recommended)
REPO_URL = "https://github.com/aditya-siraskar/PINN_FTRE"

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = "/content"
ROOT = f"{BASE}/PINN_FTRE"
if not os.path.exists(ROOT):
    !git clone -q {REPO_URL} "{ROOT}"
else:
    rc = os.system(f'git -C "{ROOT}" pull -q --ff-only')
    if rc != 0:   # usually: result files on Drive that have since been committed to GitHub
        print("WARNING: git pull failed; the Drive copy is out of date. To sync (keeps untracked files), run:")
        print(f'!git -C "{ROOT}" fetch -q origin && git -C "{ROOT}" reset -q --hard origin/main')
%cd "{ROOT}"
!pip install -q -e .
sys.path.insert(0, ROOT)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
# autoreload picks up library edits without a restart. Colab's IPython build imports the
# removed `imp` module on Python >= 3.12, so it is optional: if it is skipped, use
# Runtime > Restart session after editing pinn_tailrisk/.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload unavailable here; restart the session after editing the library")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pinn_tailrisk import plotting as P
from pinn_tailrisk.notebook import show_figures, load_metrics
P.use_style()

## 1. Method in one paragraph
Same recipe as the 1-D PINN (log density, log residual × t, start at t0 from the exact density, 48 Gauss–Hermite
nodes for the jump integral, which stays 1-D because the jump is common). New for d dimensions: inputs whitened by
the Cholesky factor of Σt; the trace term tr(Σ∇²u) is exact for d ≤ 4 and a **Hutchinson** estimate (4 Rademacher
probes) above; no box domain: collocation, the normalisation integral and portfolio VaR/ES all use one importance
proposal (Student-t mixture over 0–3 jumps, built from the PIDE coefficients only). Tail-adaptive sampling is
dropped (Stage 2c: no gain). Every portfolio is checked against its **exact** 1-D law.

Each run also reports **IS with the exact density** on the same draws: the sampling floor. PINN error above that
floor is density error.

In [ ]:
print(open('configs/stage4_pinn_nd.yaml').read())

## 2. Train one run per dimension
Resumable: finished runs are skipped, so a disconnect only costs the run in progress.
Measured on a T4: 9–14 min per run, **about 1 h for all five**. For a quick pass set `STEPS = 5000`.

In [ ]:
import pathlib
STEPS = 20000
RUNS = [(1, ""), (2, ""), (5, ""), (10, ""), (5, "exact")]   # (d, tag); "exact" forces the exact trace
for d, tag in RUNS:
    run = f"stage4_pinn_nd_d{d}" + (f"_{tag}" if tag else "")
    if pathlib.Path(f"results/{run}/metrics.json").exists():
        print("skip", run); continue
    extra = f"--trace exact --tag {tag}" if tag else ""
    !python scripts/stage4_pinn_nd.py --d {d} --steps {STEPS} {extra}

## 3. Per-dimension figures and what to check

| Figure | Pass criterion |
|---|---|
| fig01_training | all loss terms fall; tracked ES₉₉ error of the equal-weight portfolio drops below 1% |
| fig02_density_slices | PINN (blue) on the exact (dashed) along the jump direction β, including the jump shoulder, and across it |
| fig03_portfolio_es | PINN points near the grey IS floor and below 1% for every portfolio; MC (aqua) for scale |
| fig04_var_es_error | median VaR/ES error over portfolios below 1% at every α and horizon |

In [ ]:
for d in (1, 2, 5, 10):
    print(f"======== d = {d}")
    show_figures(f"stage4_pinn_nd_d{d}")

## 4. Scaling summary (accuracy, time and memory vs d; Hutchinson vs exact trace)

In [ ]:
!python scripts/stage4_summary.py
show_figures("stage4_summary")

## 5. Numbers

In [ ]:
display(pd.read_csv("results/stage4_summary.csv").style.format(precision=4))
for d in (1, 2, 5, 10):
    m = load_metrics(f"stage4_pinn_nd_d{d}")
    print(d, m["trace"], f'{m["train_time_s"]/60:.1f} min', f'{m["peak_mem_mb"]:.0f} MB', m["importance_sampling"], m["timing"]["1d"])

## 6. d = 10: reported as the breakdown point (not pursued)
At d = 10 the ES₉₉ error is 5% at 1 day and 62% at 10 days (PDE loss stalls at 0.76 vs 0.14 at d = 5; density too wide
along β). Following the plan's fallback, the project stops at d = 5 and reports d = 10 as where the method breaks down.
The cell below is a commented skeleton of the follow-up runs (exact trace; then width 256 + 40k steps) for future work.

In [ ]:
# Future work, not run in this project. Uncomment to try the d = 10 follow-up (~75 min on a T4).
# FOLLOWUP = [("exact", "--trace exact"),                                     # A: exact trace only
#             ("exact_w256_40k", "--trace exact --width 256 --steps 40000")]  # B: + capacity and steps
# for tag, flags in FOLLOWUP:
#     if pathlib.Path(f"results/stage4_pinn_nd_d10_{tag}/metrics.json").exists():
#         print("skip", tag); continue
#     !python scripts/stage4_pinn_nd.py --d 10 --tag {tag} {flags}
# !python scripts/stage4_summary.py